In [85]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import scipy.stats as stats

# 1. 데이터 불러오기

In [86]:
df = pd.read_csv('./data_for_eda/telecom_final_dataset_encoding.csv', encoding="cp949")
df.head()

,id,gender,age,year,school,income,job,phone_usage_per_m,mobile_bundle,target,...,jeonbuk,jeonnam,gyeongbuk,gyeongnam,jeju,sejong,skt,kt,lgu,mvno
0,10001,0,50,2017,4,0,0,29,0,1,...,0,0,0,0,0,0,1,0,0,0
1,10001,0,51,2018,4,0,0,45,1,1,...,0,0,0,0,0,0,0,1,0,0
2,10001,0,52,2019,4,10,1,75,1,1,...,0,0,0,0,0,0,1,0,0,0
3,10001,0,53,2020,4,10,1,41,1,0,...,0,0,0,0,0,0,0,1,0,0
4,10001,0,54,2021,4,9,1,32,1,0,...,0,0,0,0,0,0,0,1,0,0


# 2. 컬럼 추가

In [87]:
df = df.sort_values(["id", "year"]).reset_index(drop=True)
telecom_onehot_cols = ["skt", "kt", "lgu", "mvno"]

In [88]:
# 원핫 -> 단일 통신사 라벨 컬럼 만들기
df["telecom"] = df[telecom_onehot_cols].idxmax(axis=1)

In [89]:
df["prev_telecom"] = df.groupby("id")["telecom"].shift(1)

In [90]:
# run_id / run_len / prev_run_len
# 통신사가 바뀌는 지점마다 run_id +1
df["run_id"] = (
    (df["telecom"] != df["prev_telecom"])
    .groupby(df["id"])
    .cumsum()
)

# run 안에서 몇 년째인지
df["run_len"] = df.groupby(["id", "run_id"]).cumcount() + 1

# 바꾸기 직전 연속기간(직전 year의 run_len)
df["prev_run_len"] = df.groupby("id")["run_len"].shift(1).fillna(0).astype(int)

In [91]:
# 2년 이상 유지 후 변경한 경우만 이탈 = 1
df["telecom_changed_2yr"] = np.where(
    (df["prev_telecom"].notna()) &
    (df["prev_telecom"] != df["telecom"]) &
    (df["prev_run_len"] >= 2),
    1, 0).astype(int)

In [92]:
df["telecom_changed_2yr"].value_counts()

telecom_changed_2yr
0    56240
1     7086
Name: count, dtype: int64

In [93]:
cols_to_drop = ['telecom', 'prev_telecom', 'run_id', 'run_len', 'prev_run_len']
df = df.drop(columns=cols_to_drop)

In [94]:
df.head()

,id,gender,age,year,school,income,job,phone_usage_per_m,mobile_bundle,target,...,jeonnam,gyeongbuk,gyeongnam,jeju,sejong,skt,kt,lgu,mvno,telecom_changed_2yr
0,10001,0,50,2017,4,0,0,29,0,1,...,0,0,0,0,0,1,0,0,0,0
1,10001,0,51,2018,4,0,0,45,1,1,...,0,0,0,0,0,0,1,0,0,0
2,10001,0,52,2019,4,10,1,75,1,1,...,0,0,0,0,0,1,0,0,0,0
3,10001,0,53,2020,4,10,1,41,1,0,...,0,0,0,0,0,0,1,0,0,0
4,10001,0,54,2021,4,9,1,32,1,0,...,0,0,0,0,0,0,1,0,0,0


# 4. 모델 학습

In [99]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. 데이터 로드 및 시간 기준 분리
df = pd.read_csv('tel_preprocessed_2.csv')

train = df[df['year'] <= 2022]
test = df[df['year'] >= 2023]

# 2. Feature(X)와 Target(y) 설정
drop_cols = ['id', 'year', 'target']

X_train = train.drop(columns=drop_cols)
y_train = train['target']

X_test = test.drop(columns=drop_cols)
y_test = test['target']

# 3. CatBoost 모델 설정 및 학습
model = CatBoostClassifier(
    iterations=500,               # 트리 개수 (XGBoost의 n_estimators와 동일)
    learning_rate=0.05,           # 학습률
    depth=6,                      # 트리 깊이
    auto_class_weights='Balanced', # 🌟 핵심! 불균형 데이터를 알아서 맞춰줍니다.
    random_seed=42,
    verbose=100                   # 학습 과정을 100번마다 한 번씩만 깔끔하게 출력
)

print("CatBoost 학습을 시작합니다...")
model.fit(X_train, y_train)

# 4. 예측 및 평가
y_pred = model.predict(X_test)

print("\n=== CatBoost 최종 테스트 결과 ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

CatBoost 학습을 시작합니다...
0:	learn: 0.6910494	total: 13.2ms	remaining: 6.61s
100:	learn: 0.6447546	total: 928ms	remaining: 3.67s
200:	learn: 0.6362475	total: 1.83s	remaining: 2.73s
300:	learn: 0.6285755	total: 2.7s	remaining: 1.79s
400:	learn: 0.6227499	total: 3.57s	remaining: 881ms
499:	learn: 0.6178842	total: 4.44s	remaining: 0us

=== CatBoost 최종 테스트 결과 ===
Accuracy: 0.6017
Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.61      0.66      9638
           1       0.47      0.59      0.52      5608

    accuracy                           0.60     15246
   macro avg       0.59      0.60      0.59     15246
weighted avg       0.63      0.60      0.61     15246



In [100]:
from catboost import CatBoostClassifier

target = "telecom_changed_2yr"

features = [
    "age",
    "income",
    "phone_usage_per_m",
    "mobile_bundle",
    "skt",
    "kt",
    "lgu",
    "mvno",
    "year"
]

split_year = 2022

train = df[df["year"] <= split_year]
test  = df[df["year"] > split_year]

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

model = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.03,
    eval_metric="F1",
    class_weights=[1, 1.6],
    verbose=200,
    random_seed=42
)

model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=200,
    use_best_model=True
)

0:	learn: 0.0000000	test: 0.0000000	best: 0.0000000 (0)	total: 10.3ms	remaining: 15.5s
200:	learn: 0.0120090	test: 0.0090472	best: 0.0098067 (188)	total: 1.98s	remaining: 12.8s
400:	learn: 0.0251496	test: 0.0193452	best: 0.0193587 (396)	total: 3.94s	remaining: 10.8s
600:	learn: 0.0376883	test: 0.0237290	best: 0.0237290 (599)	total: 5.93s	remaining: 8.88s
800:	learn: 0.0496543	test: 0.0258529	best: 0.0258529 (800)	total: 8.03s	remaining: 7s
1000:	learn: 0.0614502	test: 0.0265328	best: 0.0272786 (956)	total: 10.2s	remaining: 5.1s
1200:	learn: 0.0760105	test: 0.0314946	best: 0.0315452 (1119)	total: 12.7s	remaining: 3.16s
1400:	learn: 0.0871004	test: 0.0314084	best: 0.0321491 (1241)	total: 15.4s	remaining: 1.09s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.03214905472
bestIteration = 1241

Shrink model to first 1242 iterations.


# 99. csv 변환

In [96]:
df.to_csv('tel_preprocessed_ver2.csv', index=False, encoding='utf-8-sig')